# AksharDrishti: speed benchmark (T4)
Run all. Cell 2 installs and restarts once; then Run all again. Needs a T4 GPU and the `HF_TOKEN` secret. Tests which settings cut the 27 s/page parser time without hurting CER.

In [ ]:
# 1. GPU
!nvidia-smi

In [ ]:
# 2. Install everything once, then restart automatically (Run all again after the restart)
import os
if not os.path.exists("/content/.deps_ok"):
    !pip install -q -U "transformers>=5.7" "accelerate>=1.1" "huggingface_hub>=1.0" "pillow>=10.0.1,<13" rapidfuzz
    !apt-get -qq install -y fonts-noto-core > /dev/null
    open("/content/.deps_ok", "w").write("ok")
    print("Installed. Restarting the session - then use Runtime > Run all again.")
    os.kill(os.getpid(), 9)
print("dependencies already installed")

In [ ]:
# 3. Log in to Hugging Face and download the model + weights (~1.8 GB)
from google.colab import userdata
from huggingface_hub import login, snapshot_download

login(userdata.get("HF_TOKEN"))
repo = snapshot_download("bodhan-ai/indic-ocr", local_dir="/content/indic-ocr")
print("Downloaded to", repo)

In [ ]:
# 4. Load the model (once per session)
import sys, torch
sys.path.insert(0, repo)
from indic_ocr import IndicOCR

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
parser = IndicOCR.from_pretrained(repo)   # add table_format="markdown" for Markdown tables

In [ ]:
# G1. Build the synthetic benchmark: realistic government pages with exact ground truth
import random, re, os, json, glob, io, math, urllib.request
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter, features

SYN = "/content/synthetic"
N_PER_LANG = 10           # 10 Marathi + 10 Hindi + 10 English = 30 pages
random.seed(2026); np.random.seed(2026)
os.makedirs(SYN, exist_ok=True)
print("complex script shaping (raqm):", features.check_feature("raqm"))

def font_path(name):
    hits = glob.glob(f"/usr/share/fonts/**/{name}", recursive=True)
    if not hits: raise FileNotFoundError(name)
    return hits[0]
FP = {("dev", False): font_path("NotoSansDevanagari-Regular.ttf"), ("dev", True): font_path("NotoSansDevanagari-Bold.ttf"),
      ("lat", False): font_path("NotoSans-Regular.ttf"), ("lat", True): font_path("NotoSans-Bold.ttf")}
CURSIVE = []
for url in ["https://github.com/google/fonts/raw/main/ofl/greatvibes/GreatVibes-Regular.ttf",
            "https://github.com/google/fonts/raw/main/ofl/allura/Allura-Regular.ttf",
            "https://github.com/google/fonts/raw/main/ofl/sacramento/Sacramento-Regular.ttf"]:
    try:
        path = "/content/" + url.rsplit("/", 1)[1]
        if not os.path.exists(path): urllib.request.urlretrieve(url, path)
        ImageFont.truetype(path, 40); CURSIVE.append(path)
    except Exception as e:
        print("cursive font not available:", url, e)
print("cursive fonts:", len(CURSIVE))

_fc = {}
def F(size, bold=False, dev=False):
    k = (size, bold, dev)
    if k not in _fc:
        kw = {"layout_engine": ImageFont.Layout.RAQM} if dev and features.check_feature("raqm") else {}
        _fc[k] = ImageFont.truetype(FP[("dev" if dev else "lat", bold)], size, **kw)
    return _fc[k]

DEVCH = re.compile(r"[ऀ-ॿ‌‍]")
def runs(text):
    out, cur, mode = [], "", None
    for ch in text:
        d = mode if ch == " " else bool(DEVCH.match(ch))
        if mode is None: mode = bool(d)
        if ch != " " and d != mode:
            out.append((cur, mode)); cur, mode = ch, d
        else:
            cur += ch
    if cur: out.append((cur, mode))
    return out
def tw(text, size, bold=False):
    return sum(F(size, bold, dv).getlength(s) for s, dv in runs(text))
def put(d, x, base, text, size, bold=False, fill=(25, 25, 25)):
    for s, dv in runs(text):
        d.text((x, base), s, font=F(size, bold, dv), fill=fill, anchor="ls")
        x += F(size, bold, dv).getlength(s)
    return x

DEVDIG = str.maketrans("0123456789", "०१२३४५६७८९")
LANG = {
 "mr": dict(head=["महाराष्ट्र शासन", "महसूल व वन विभाग", "तहसील कार्यालय, {place}"],
            titles=["जाहीर सूचना", "उत्पन्नाचा दाखला", "फेरफार नोंद सूचना"],
            para="याद्वारे सर्व संबंधितांना कळविण्यात येते की खाली नमूद केलेल्या जमिनीच्या फेरफार नोंदीबाबत हरकती असल्यास त्या पंधरा दिवसांच्या आत या कार्यालयात सादर कराव्यात. मुदतीनंतर आलेल्या हरकतींचा विचार केला जाणार नाही.",
            name="अर्जदाराचे नाव", date="दिनांक", aad="आधार क्रमांक", pan="पॅन क्रमांक", phone="मोबाईल क्रमांक",
            pin="पिन कोड", fee="शुल्क", cur="रु.", land="जमीन", survey="गट नं.", email=None,
            table=["अ.क्र.", "नाव", "भूखंड क्र.", "क्षेत्र (हे.)"], desig="तहसीलदार",
            names=["रमेश पाटील", "सुनीता जाधव", "गणेश कुलकर्णी", "अनिता देशमुख", "प्रकाश शिंदे"],
            places=["नाशिक", "पुणे", "नागपूर", "सातारा"]),
 "hi": dict(head=["भारत सरकार", "राजस्व विभाग", "जिला कार्यालय, {place}"],
            titles=["सूचना", "आय प्रमाण पत्र", "भूमि अभिलेख सूचना"],
            para="सभी संबंधित व्यक्तियों को सूचित किया जाता है कि नीचे दी गई भूमि के अभिलेख में परिवर्तन के संबंध में यदि कोई आपत्ति हो तो वह पंद्रह दिनों के भीतर इस कार्यालय में प्रस्तुत करें। निर्धारित अवधि के बाद प्राप्त आपत्तियों पर विचार नहीं किया जाएगा।",
            name="आवेदक का नाम", date="दिनांक", aad="आधार संख्या", pan="पैन संख्या", phone="मोबाइल नंबर",
            pin="पिन कोड", fee="शुल्क", cur="रुपये", land="भूमि", survey="सर्वे नं.", email=None,
            table=["क्र.", "नाम", "भूखंड सं.", "क्षेत्रफल (हे.)"], desig="तहसीलदार",
            names=["राजेश कुमार", "सीमा वर्मा", "अमित शर्मा", "पूजा सिंह", "विकास यादव"],
            places=["लखनऊ", "भोपाल", "जयपुर", "पटना"]),
 "en": dict(head=["Government of Maharashtra", "Revenue and Forest Department", "Office of the Tahsildar, {place}"],
            titles=["PUBLIC NOTICE", "INCOME CERTIFICATE", "MUTATION ENTRY NOTICE"],
            para="All concerned persons are hereby informed that objections, if any, regarding the mutation entry of the land mentioned below should be submitted to this office within fifteen days. Objections received after the due period will not be considered.",
            name="Applicant Name", date="Date", aad="Aadhaar No.", pan="PAN", phone="Mobile No.",
            pin="PIN Code", fee="Fee", cur="Rs.", land="Land", survey="Survey No.", email="E-mail",
            table=["Sr.", "Name", "Plot", "Area (ha)"], desig="Tahsildar",
            names=["Ramesh Patil", "Sunita Jadhav", "Amit Sharma", "Pooja Singh", "Vikas Yadav"],
            places=["Nashik", "Pune", "Nagpur", "Satara"]),
}
SIGN_NAMES = ["R. Patil", "S. Kulkarni", "A. Deshmukh", "V. Sharma", "P. Shinde", "M. Joshi", "K. Rao"]

def rand_values(lang):
    v = {}
    v["Date"] = random.choice(["{:02d}/{:02d}/20{}", "{:02d}.{:02d}.20{}", "{:02d}-{:02d}-20{}"]).format(
        random.randint(1, 28), random.randint(1, 12), random.randint(18, 26))
    if random.random() < 0.6: v["Aadhaar"] = f"{random.randint(2,9)}{random.randint(0,999):03d} {random.randint(0,9999):04d} {random.randint(0,9999):04d}"
    if random.random() < 0.5: v["PAN"] = "".join(random.choice("ABCDEFGHJKLMNPQRSTUVWXYZ") for _ in range(5)) + f"{random.randint(0,9999):04d}" + random.choice("ABCDEFGHJKLMNPQRSTUVWXYZ")
    if random.random() < 0.7: v["Phone"] = f"{random.randint(6,9)}{random.randint(0,999999999):09d}"
    if random.random() < 0.6: v["PIN code"] = f"{random.randint(1,8)}{random.randint(0,99999):05d}"
    if random.random() < 0.6: v["Amount"] = random.choice(["500", "1,500", "2,750", "10,000", "25"])
    if random.random() < 0.7: v["Survey"] = random.choice([f"{random.randint(1,300)}", f"{random.randint(1,300)}/{random.randint(1,9)}"])
    if lang == "en" and random.random() < 0.6:
        v["Email"] = random.choice(["tahsil", "office", "revenue", "collector"]) + f"{random.randint(1,99)}@maharashtra.gov.in"
    return v

def make_signature(name):
    ink = random.choice([(20, 35, 120), (20, 35, 120), (25, 25, 25)])
    im = Image.new("RGBA", (520, 200), (0, 0, 0, 0)); d = ImageDraw.Draw(im)
    if CURSIVE and random.random() < 0.7:
        f = ImageFont.truetype(random.choice(CURSIVE), random.randint(64, 86))
        d.text((20, 110), name, font=f, fill=ink + (255,), anchor="ls")
        x1 = 20 + f.getlength(name)
        pts = [(30, 135), (x1 * 0.5, 145 + random.randint(-8, 8)), (x1 + 20, 125)]
        d.line(pts, fill=ink + (255,), width=3, joint="curve")
    else:                                   # free-hand scribble
        x, y = 30.0, 110.0
        pts = []
        for i in range(random.randint(40, 70)):
            x += random.uniform(3, 9); y += random.uniform(-14, 14); y = min(max(y, 40), 170)
            pts.append((x, y))
        d.line(pts, fill=ink + (255,), width=3, joint="curve")
        d.line([(30, 150), (x, 140)], fill=ink + (255,), width=2)
    im = im.rotate(random.uniform(-8, 8), expand=True, resample=Image.BICUBIC)
    return im.crop(im.getbbox())

def arc_text(img, text, cx, cy, r, color, size):
    f = F(size, True, False)
    total = sum(f.getlength(c) for c in text)
    ang = -math.pi / 2 - (total / r) / 2
    for c in text:
        w = f.getlength(c)
        a = ang + (w / 2) / r
        g = Image.new("RGBA", (size * 2, size * 2), (0, 0, 0, 0))
        ImageDraw.Draw(g).text((size, size), c, font=f, fill=color, anchor="mm")
        g = g.rotate(-math.degrees(a + math.pi / 2), resample=Image.BICUBIC)
        img.alpha_composite(g, (int(cx + r * math.cos(a) - size), int(cy + r * math.sin(a) - size)))
        ang += w / r

def make_stamp(lang, place, black):
    color = (35, 35, 35) if black else random.choice([(35, 70, 190), (115, 45, 165), (190, 35, 45)])
    c = color + (255,)
    if random.random() < 0.7:               # round seal
        S = 300; im = Image.new("RGBA", (S, S), (0, 0, 0, 0)); d = ImageDraw.Draw(im)
        d.ellipse([6, 6, S - 6, S - 6], outline=c, width=6); d.ellipse([50, 50, S - 50, S - 50], outline=c, width=2)
        arc_text(im, "OFFICE OF THE TAHSILDAR", S / 2, S / 2, 122, c, 22)
        star = [(S/2 + 22 * math.cos(math.pi/2 + k*math.pi/5) * (1 if k % 2 == 0 else 0.45),
                 S/2 - 22 * math.sin(math.pi/2 + k*math.pi/5) * (1 if k % 2 == 0 else 0.45)) for k in range(10)]
        d.polygon(star, fill=c)
        lines = [LANG[lang]["desig"], place]
        for i, t in enumerate(lines):
            put(d, S/2 - tw(t, 24, True)/2, S/2 + 52 + i * 30, t, 24, True, c)
    else:                                   # rectangular stamp
        W_, H_ = 360, 170; im = Image.new("RGBA", (W_, H_), (0, 0, 0, 0)); d = ImageDraw.Draw(im)
        d.rectangle([4, 4, W_ - 4, H_ - 4], outline=c, width=5); d.rectangle([14, 14, W_ - 14, H_ - 14], outline=c, width=2)
        for i, t in enumerate([LANG[lang]["desig"], place, "SEAL"]):
            put(d, W_/2 - tw(t, 30, True)/2, 58 + i * 40, t, 30, True, c)
    a = np.array(im)                        # faded, uneven ink
    keep = (np.random.rand(*a.shape[:2]) > random.uniform(0.05, 0.25)).astype(np.float32)
    a[..., 3] = (a[..., 3] * keep * random.uniform(0.6, 0.9)).astype(np.uint8)
    im = Image.fromarray(a).rotate(random.uniform(-25, 25), expand=True, resample=Image.BICUBIC)
    return im.crop(im.getbbox())

def paste(page, im, x, y):
    page.alpha_composite(im, (int(x), int(y)))
    b = im.getchannel("A").point(lambda v: 255 if v > 40 else 0).getbbox()
    W_, H_ = page.size
    return [max(x + b[0], 0), max(y + b[1], 0), min(x + b[2], W_), min(y + b[3], H_)]

def make_page(idx, lang, scan):
    L = LANG[lang]; place = random.choice(L["places"]); name = random.choice(L["names"])
    devdig = lang == "mr" and random.random() < 0.4
    num = (lambda s: s.translate(DEVDIG)) if devdig else (lambda s: s)
    W, H = 1240, 1754
    page = Image.new("RGBA", (W, H), (252, 251, 247, 255)); d = ImageDraw.Draw(page)
    gt = {"page": f"syn_{idx:03d}", "lang": lang, "scan": scan, "devanagari_digits": devdig,
          "signatures": [], "stamps": [], "fields": [], "sensitive": []}
    y = 120
    for i, h in enumerate(L["head"]):
        t = h.format(place=place); s = [36, 30, 30][i]
        put(d, W/2 - tw(t, s, True)/2, y, t, s, True); y += int(s * 1.6)
    d.line([(90, y), (W - 90, y)], fill=(40, 40, 40), width=3); y += 80
    t = random.choice(L["titles"]); put(d, W/2 - tw(t, 40, True)/2, y, t, 40, True); y += 90
    words, line = L["para"].split(), ""
    for w in words + [None]:
        if w is None or tw(line + " " + w, 28) > W - 200:
            put(d, 100, y, line.strip(), 28); y += 48; line = ""
        if w: line += " " + w
    y += 30
    vals = rand_values(lang)
    rows = [(L["name"], name, None)]
    rows.append((L["date"], num(vals["Date"]), "Date"))
    if "Aadhaar" in vals: rows.append((L["aad"], num(vals["Aadhaar"]), "Aadhaar"))
    if "PAN" in vals: rows.append((L["pan"], vals["PAN"], "PAN"))
    if "Phone" in vals: rows.append((L["phone"], num(vals["Phone"]), "Phone"))
    if "PIN code" in vals: rows.append((L["pin"], num(vals["PIN code"]), "PIN code"))
    if "Amount" in vals: rows.append((L["fee"], f"{L['cur']} {num(vals['Amount'])}", "Amount"))
    if "Survey" in vals: rows.append((L["land"], f"{L['survey']} {num(vals['Survey'])}", "Survey / Gat no."))
    if "Email" in vals: rows.append((L["email"], vals["Email"], "Email"))
    for label, value, field in rows:
        x = put(d, 100, y, f"{label} : ", 28)
        x1 = put(d, x, y, value, 28)
        if field:
            gt["fields"].append({"field": field, "value": value})
            if field in ("Aadhaar", "PAN"):
                gt["sensitive"].append({"field": field, "value": value, "box": [x, y - 30, x1, y + 10]})
        y += 50
    if random.random() < 0.5:               # small table without field keywords
        y += 20; cols = [100, 220, 620, 860, W - 100]
        data = [L["table"]] + [[num(str(i)), random.choice(L["names"]), num(random.choice(["12", "7/1", "45/2", "118"])),
                                num(random.choice(["0.80", "1.20", "2.05"]))] for i in range(1, 4)]
        for r, row in enumerate(data):
            for ci, cell in enumerate(row):
                put(d, cols[ci] + 12, y + 40, cell, 26, r == 0)
            d.line([(cols[0], y), (cols[-1], y)], fill=(40, 40, 40), width=2); y += 56
        d.line([(cols[0], y), (cols[-1], y)], fill=(40, 40, 40), width=2)
        for cx in cols: d.line([(cx, y - 56 * len(data)), (cx, y)], fill=(40, 40, 40), width=2)
    # signature block
    by = H - 260
    put(d, W - 420, by + 120, L["desig"], 28, True); put(d, W - 420, by + 165, place, 26)
    if random.random() < 0.85:
        sig = make_signature(random.choice(SIGN_NAMES))
        gt["signatures"].append(paste(page, sig, W - 440 + random.randint(-20, 40), by + 90 - sig.height))
    if random.random() < 0.85:
        black = random.random() < 0.25
        st = make_stamp(lang, place, black)
        sx = random.choice([W - 760, W - 560, W - 420]); sy = by + random.randint(-120, 20) - st.height // 3
        gt["stamps"].append({"box": paste(page, st, sx, sy), "ink": "black" if black else "colour"})
    img = page.convert("RGB")
    if scan == "photo":                     # shadow, blur, noise, JPEG
        a = np.asarray(img).astype(np.float32)
        g = np.linspace(random.uniform(0.65, 0.8), 1.0, W)[None, :, None]
        if random.random() < 0.5: g = g[:, ::-1]
        a = a * g * np.array([1.0, 0.98, 0.92]) + np.random.normal(0, 6, a.shape)
        img = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(random.uniform(0.7, 1.2)))
        buf = io.BytesIO(); img.save(buf, "JPEG", quality=55); img = Image.open(buf).convert("RGB")
        path = f"{SYN}/{gt['page']}.jpg"; img.save(path, quality=90)
    else:
        a = np.asarray(img).astype(np.float32) + np.random.normal(0, 2, (H, W, 3))
        img = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
        path = f"{SYN}/{gt['page']}.png"; img.save(path)
    gt["path"] = path
    return gt

GT = []
for i, lang in enumerate(["mr"] * N_PER_LANG + ["hi"] * N_PER_LANG + ["en"] * N_PER_LANG):
    GT.append(make_page(i + 1, lang, "photo" if i % 2 else "scan"))
json.dump(GT, open(f"{SYN}/ground_truth.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(len(GT), "pages |", sum(len(g["signatures"]) for g in GT), "signatures |",
      sum(len(g["stamps"]) for g in GT), "stamps |", sum(len(g["fields"]) for g in GT), "fields")
from IPython.display import display
display(Image.open(GT[0]["path"]).resize((620, 877))); display(Image.open(GT[21]["path"]).resize((620, 877)))

In [ ]:
# P1. Reference text (same seed -> same pages) + scoring helpers
import re, unicodedata, time, statistics, json, random, html as _h, numpy as np, torch
from rapidfuzz.distance import Levenshtein
DEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")
def clean(s): return (s or "").replace("▁", " ")
def to_plain(text):
    t = clean(text)
    t = re.sub(r"<br\s*/?>", " ", t); t = re.sub(r"</t[dh]>", " ", t); t = re.sub(r"</tr>", "\n", t)
    return _h.unescape(re.sub(r"<[^>]+>", "", t)).replace("$", "").strip()

GT_OLD = json.load(open(f"{SYN}/ground_truth.json", encoding="utf-8"))
_LOG = []
if "_put_orig" not in globals():
    _put_orig = put
    def put(d, x, base, text, size, bold=False, fill=(25, 25, 25)):
        if d.im.size == (1240, 1754):
            _LOG.append((round(base), x, text))
        return _put_orig(d, x, base, text, size, bold, fill)
random.seed(2026); np.random.seed(2026)
REF, same = {}, 0
for i, lang in enumerate(["mr"] * N_PER_LANG + ["hi"] * N_PER_LANG + ["en"] * N_PER_LANG):
    _LOG.clear()
    g = make_page(i + 1, lang, "photo" if i % 2 else "scan")
    same += g["fields"] == GT_OLD[i]["fields"] and g["stamps"] == GT_OLD[i]["stamps"]
    rows = {}
    for base, x, text in _LOG: rows.setdefault(base, []).append((x, text))
    REF[g["page"]] = "\n".join(" ".join(t.strip() for _, t in sorted(rows[b])) for b in sorted(rows))
print(f"reference pages identical to the evaluated ones: {same}/{len(GT_OLD)}")

def normtxt(s):
    s = unicodedata.normalize("NFC", (s or "").replace("▁", " ")).translate(DEV_DIGITS)
    s = re.sub(r"[\u200b\u200c\u200d\ufffd]", "", s)
    return " ".join(s.split())
def score(texts, pages):
    ce = cn = we = wn = 0
    for p in pages:
        r, h = normtxt(REF[p]), normtxt(texts[p])
        ce += Levenshtein.distance(r, h); cn += len(r)
        we += Levenshtein.distance(r.split(), h.split()); wn += len(r.split())
    return ce / cn, we / wn
ALL = [g["page"] for g in GT_OLD]
PATHS = {g["page"]: g["path"] for g in GT_OLD}
SUB = ALL[::3]          # 10 pages: mixed language, scan and photo
print("subset:", SUB)

In [ ]:
# P2. Configurable recognizer + timed runner (patches the instance only; package files untouched)
rec = parser.recognizer.backend          # HfRecognizer (loads the 1.7 GB model)
print("model dtype:", next(rec.model.parameters()).dtype, "| attn:", getattr(rec.model.config, "_attn_implementation", "?"))
TOK = []

def make_transcribe(bs, max_tokens, sort):
    def transcribe(requests):
        order = list(range(len(requests)))
        if sort:
            order.sort(key=lambda i: requests[i].image.size[0] * requests[i].image.size[1])
        texts = [None] * len(requests)
        for k in range(0, len(order), bs):
            idx = order[k:k + bs]; chunk = [requests[i] for i in idx]
            inputs = rec.processor(text=[rec._prompt(r.prompt) for r in chunk], images=[r.image for r in chunk],
                                   padding=True, return_tensors="pt").to(rec.model.device)
            L = inputs["input_ids"].shape[-1]
            with torch.inference_mode():
                out = rec.model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, use_cache=True,
                                         eos_token_id=rec.eos_token_id, pad_token_id=rec.pad_token_id)
            gen = out[:, L:]
            for j, i in enumerate(idx):
                texts[i] = rec.processor.decode(gen[j], skip_special_tokens=True)
                TOK.append(int((gen[j] != rec.pad_token_id).sum()))
        return [t.strip() for t in texts]
    return transcribe

def run_variant(name, pages, dtype="bfloat16", bs=8, max_tokens=2048, sort=False, warm=True):
    rec.model.to(getattr(torch, dtype))
    rec.transcribe = make_transcribe(bs, max_tokens, sort)
    if warm:
        parser.parse(PATHS[pages[0]])
    TOK.clear(); torch.cuda.reset_peak_memory_stats()
    texts, tl, tr = {}, [], []
    for p in pages:
        torch.cuda.synchronize(); t0 = time.time()
        lay = parser.layout.detect(PATHS[p]); torch.cuda.synchronize(); t1 = time.time()
        page = parser.recognizer.run(PATHS[p], lay); torch.cuda.synchronize(); t2 = time.time()
        tl.append(t1 - t0); tr.append(t2 - t1)
        blocks = sorted(page.as_record()["blocks"], key=lambda b: b["order"])
        texts[p] = "\n".join(to_plain(b.get("text")) for b in blocks if b.get("text"))
    cer, wer = score(texts, pages)
    res = dict(name=name, dtype=dtype, bs=bs, max_tokens=max_tokens, sort=sort, pages=len(pages),
               layout_s=statistics.mean(tl), recog_s=statistics.mean(tr), total_s=statistics.mean(tl) + statistics.mean(tr),
               cer=cer, wer=wer, peak_gb=torch.cuda.max_memory_allocated() / 1e9,
               tok_mean=statistics.mean(TOK) if TOK else 0, tok_max=max(TOK) if TOK else 0, n_blocks=len(TOK))
    print(f"{name:30s} total {res['total_s']:6.2f}s (layout {res['layout_s']:.2f} + recog {res['recog_s']:.2f}) "
          f"CER {cer:.3f} WER {wer:.3f} blocks {res['n_blocks']} tok mean/max {res['tok_mean']:.0f}/{res['tok_max']} peak {res['peak_gb']:.2f}GB", flush=True)
    return res, texts
print("ready")

In [ ]:
# P3. Screening on 10 pages: one change at a time, cumulative where it helps
R = []
VARIANTS = [
    dict(name="baseline bf16 bs8 mt2048", dtype="bfloat16", bs=8,  max_tokens=2048, sort=False),
    dict(name="fp16",                     dtype="float16",  bs=8,  max_tokens=2048, sort=False),
    dict(name="fp16 + sort",              dtype="float16",  bs=8,  max_tokens=2048, sort=True),
    dict(name="fp16 + sort bs16",         dtype="float16",  bs=16, max_tokens=2048, sort=True),
    dict(name="fp16 + sort bs16 mt1024",  dtype="float16",  bs=16, max_tokens=1024, sort=True),
    dict(name="fp16 + sort bs16 mt512",   dtype="float16",  bs=16, max_tokens=512,  sort=True),
    dict(name="fp16 + sort bs32 mt1024",  dtype="float16",  bs=32, max_tokens=1024, sort=True),
]
for v in VARIANTS:
    try:
        res, texts = run_variant(v.pop("name"), SUB, **v)
    except Exception as e:
        print("FAILED", v, repr(e)[:300]); torch.cuda.empty_cache(); continue
    R.append(res)
json.dump(R, open("/content/speed_screen.json", "w"))
print("\nSCREEN_JSON=" + json.dumps([{k: (round(x, 4) if isinstance(x, float) else x) for k, x in r.items()} for r in R]))

In [ ]:
# P4. Fastest variant with CER within +0.005 of baseline, then baseline vs best on all 30 pages
base = R[0]
ok = [r for r in R[1:] if r["cer"] <= base["cer"] + 0.005]
best = min(ok, key=lambda r: r["total_s"]) if ok else None
print("best variant:", best["name"] if best else "none within CER tolerance")
FULL = []
if best:
    for r in (R[0], best):
        res, _ = run_variant("FULL " + r["name"], ALL, dtype=r["dtype"], bs=r["bs"], max_tokens=r["max_tokens"], sort=r["sort"])
        FULL.append(res)
    print("\nFULL_JSON=" + json.dumps([{k: (round(x, 4) if isinstance(x, float) else x) for k, x in r.items()} for r in FULL]))
    print(f"speedup: {FULL[0]['total_s'] / FULL[1]['total_s']:.2f}x")